# Reg 级计算：VF 函数与寄存器路径

Reg 计算是 Pro 在 Vector 侧提供的高性能矢量计算方式：数据从 UB 加载进 **Register File**，将适合的计算链保留在寄存器中，减少 UB 访问；链长度、寄存器压力与编译结果仍会影响实际收益。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 为什么需要 Reg 计算

Tile 计算以 UB Tile 为载体，多个矢量操作串联时，中间结果通常要**写回 UB 再读出**。计算链越长，UB 读写往返越多——带宽压力和 Bank 冲突概率随之上升。

Reg 计算把一段连续计算留在寄存器里：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">维度</th><th style="text-align:left;">Tile 计算</th><th style="text-align:left;">Reg 计算</th></tr>
<tr><td>数据载体</td><td>UB 中的 Tile</td><td>Register File 中的 RegTensor / MaskReg</td></tr>
<tr><td>中间结果</td><td>通常写回 UB</td><td>由后续 vf.* 操作直接消费</td></tr>
<tr><td>接口前缀</td><td>pl.add、pl.sub、pl.sum …</td><td>vf.add、vf.sub、vf.reduce_* …</td></tr>
<tr><td>适用场景</td><td>通用矢量计算、快速实现</td><td>连续计算链、降 UB 往返的高性能场景</td></tr>
</table>


## 2. 硬件与内存路径

Reg 计算涉及的硬件：**Reg 矢量执行单元**（从 Register File 读操作数、写回结果）、**DMA 单元**（UB 与 Register File 间搬运）、**Aux Scalar**（VF 域内标量计算）。

Register File 位于 UB 之上，**不能直接从 GM 加载或写回 GM**。完整数据路径：

```text
GM → UB → Register File → UB → GM
```

各段接口对应：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">数据路径</th><th style="text-align:left;">PyPTO Pro 表达</th></tr>
<tr><td>GM → UB</td><td>pl.load / pl.load_tile</td></tr>
<tr><td>UB → Register File</td><td>vf.load / vf.load_align / vf.load_unalign</td></tr>
<tr><td>Register File 内计算</td><td>vf.add、vf.mul、vf.reduce_sum …</td></tr>
<tr><td>Register File → UB</td><td>vf.store / vf.store_align / vf.store_unalign</td></tr>
<tr><td>UB → GM</td><td>pl.store / pl.store_tile</td></tr>
</table>


## 3. VF 函数

Reg 计算通过 `@pl.vector_function` 声明的 **VF 函数**表达：函数体隐式处于 VF 执行域，内部用 `vf.*` 操作寄存器；Tile 参数的类型由调用点推导。


In [ ]:
import pypto_pro.language as pl
from pypto_pro.language import Vf as vf


@pl.vector_function
def add_vf(src_a, src_b, dst):
    preg = vf.create_mask(pattern=pl.MaskPattern.ALL, dtype=pl.DT_FP32)
    reg_a = vf.load_align(src_a, 0)     # UB -> 寄存器
    reg_b = vf.load_align(src_b, 0)
    reg_out = vf.add(reg_a, reg_b, preg)  # 寄存器内计算
    vf.store_align(dst, reg_out, preg)   # 寄存器 -> UB

这里一个 FP32 寄存器处理 64 个元素，恰好覆盖 [1,64]；更大的 Tile 需要按行/分段加载。`vf.reduce_sum` 也只归约一个寄存器，结果在 lane 0，不能直接当作整行或整块 Tile 的归约。

【待 950 验证：官方文档 VF 函数示例】

外层 JIT Kernel 负责 GM 与 UB 的搬运与跨 Pipe 同步，在 `section_vector()` 中调用 VF 函数——结构与第 1 章"外层 Kernel + SIMT 函数"同构，但这里调用的是 VF 函数：


In [ ]:
@pl.jit(auto_mutex=True)
def add_kernel(a: pl.Tensor[[1, 64], pl.DT_FP32],
               b: pl.Tensor[[1, 64], pl.DT_FP32],
               out: pl.Tensor[[1, 64], pl.DT_FP32]):
    tt = pl.TileType(shape=[1, 64], dtype=pl.DT_FP32,
                     target_memory=pl.MemorySpace.Vec)
    a_group = pl.make_tile_group(type=tt, addrs=0x000, mutex_ids=[0])
    b_group = pl.make_tile_group(type=tt, addrs=0x100, mutex_ids=[1])
    out_group = pl.make_tile_group(type=tt, addrs=0x200, mutex_ids=[2])

    with pl.section_vector():
        tile_a = a_group.current()
        tile_b = b_group.current()
        tile_out = out_group.current()
        pl.load(tile_a, a, [0, 0])
        pl.load(tile_b, b, [0, 0])
        add_vf(tile_a, tile_b, tile_out)   # 调用 VF 函数完成寄存器计算
        pl.store(out, tile_out, [0, 0])

Host 侧构造与校验（显式单核，待 950 验证）：


In [ ]:
device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)
a = torch.randn((1, 64), dtype=torch.float32, device=device)
b = torch.randn_like(a)
out = torch.empty_like(a)
add_kernel[None, 1](a, b, out)
torch.npu.synchronize()
torch.testing.assert_close(out, a + b, rtol=1e-6, atol=1e-6)

【待 950 验证：VF、Kernel 与 Host 的完整 [1,64] 加法示例】


## 4. VF 执行域规则（易错点）

vf.* 只能在 VF 函数内使用（放进 section_vector 里也必须经由 VF 函数）。VF 函数**可以**：

- 调用其他 VF 函数；
- 使用受支持的标量表达式、`pl.range` 循环、标量 `pl.min/pl.max/pl.const`；浮点标量除法须在 VF 外预先计算，例如把 `1.0/256.0` 写成常量 `0.00390625`，寄存器间除法则使用 `vf.div`；
- 使用 `pl.DT_*` 与枚举常量；
- 对 Tile 参数做 `tile + offset` 线性元素偏移（单位是元素，只形成指针表达式，不创建新 Tile）。

VF 函数**不能**：

- 调用其他 `pl.*` 操作（Tile 操作、同步操作、`section_vector()`/`section_cube()`）；
- 推进 TileGroup 游标——`next()/current()/previous()` 产生标量运算，**必须在调用侧选好 Tile 再作参数传入**；
- 切片 Tile（`tile[a:b, c:d]`）——需要子区域时，在调用侧切好再传入；
- 指望 auto_mutex 兜底 VF 内 UB 的写后读/写后写——这类局部依赖要按接口要求用 `vf.mem_bar` 手动指定。

违反执行域限制时，前端解析器会在对应调用处报错。这条规则的记忆方式：**VF 函数只管寄存器里的活，UB 和游标的事在外面办**。


### 一行 256 个元素如何落到寄存器

FP32 每个寄存器有 64 个元素，因此第 r 行需 4 次加载，列起点为 0、64、128、192。把每个片段平方后逐 lane 相加，再做一次 64-lane reduce_sum，等价于求该行 256 个平方值之和。归约输出只有 lane 0 有效，需用 `vf.full(lane0,mask)` 广播后才能乘回四份输入。

如果只加载偏移 0，会把 64 个元素的归约误当成整行；如果忘记广播，只有输出的一小部分可能正确。保留 x0～x3 可以避免为了归一化而再次加载输入，但也延长了它们的寄存器生命周期。下面是完整 VF 计算段，外层搬运与 Host 验收见本章实践。


In [ ]:
@pl.vector_function
def rms_registers(tx, ty):
    mask = vf.create_mask(pattern=pl.MaskPattern.ALL, dtype=pl.DT_FP32)
    inv_n = vf.full(0.00390625, mask, dtype=pl.DT_FP32)  # 1/256；VF 域内不做标量除法
    eps = vf.full(1e-5, mask, dtype=pl.DT_FP32)
    one = vf.full(1.0, mask, dtype=pl.DT_FP32)
    for row in pl.range(0, 8):
        x0 = vf.load_align(tx, [row, 0])
        x1 = vf.load_align(tx, [row, 64])
        x2 = vf.load_align(tx, [row, 128])
        x3 = vf.load_align(tx, [row, 192])
        s0 = vf.mul(x0, x0, mask)          # 逐片段平方
        s1 = vf.mul(x1, x1, mask)
        s2 = vf.mul(x2, x2, mask)
        s3 = vf.mul(x3, x3, mask)
        s01 = vf.add(s0, s1, mask)         # 片段内累加
        s23 = vf.add(s2, s3, mask)
        sums = vf.add(s01, s23, mask)
        lane0 = vf.reduce_sum(sums, mask)  # 只有 lane 0 有效
        total = vf.full(lane0, mask)       # 广播回全部 lane
        mean = vf.mul(total, inv_n, mask)
        stabilized = vf.add(mean, eps, mask)
        std = vf.sqrt(stabilized, mask)
        rstd = vf.div(one, std, mask)
        y0 = vf.mul(x0, rstd, mask)        # 归一化
        y1 = vf.mul(x1, rstd, mask)
        y2 = vf.mul(x2, rstd, mask)
        y3 = vf.mul(x3, rstd, mask)
        vf.store_align(ty, y0, mask, [row, 0])
        vf.store_align(ty, y1, mask, [row, 64])
        vf.store_align(ty, y2, mask, [row, 128])
        vf.store_align(ty, y3, mask, [row, 192])


## 5. 课后练习

1. （填空）Register File 不能直接与 GM 交互，完整数据路径是 GM → ______ → ______ → ______ → GM。
2. （选择）以下哪句代码放在 VF 函数内会触发前端报错？
   A. `reg = vf.load_align(src, 0)`<br/>
   B. `tile = group.next()`<br/>
   C. `vf.store_align(dst, reg, preg)`
3. （实践）把 add_vf 扩展为 `add_relu_vf`（相加后接 ReLU），寄存器内需要新增哪步？中间结果需要回 UB 吗？

In [ ]:
!cat ./answer/03.02_answer.txt

## 6. 本节小结

- Reg 计算 = **寄存器内完成计算链**，减少 UB 往返；路径 GM→UB→RF→UB→GM；
- `@pl.vector_function` 定义 VF 函数，`vf.load_align/vf.add/vf.store_align` 三段式；
- **执行域规则**：vf.* 只在 VF 函数内；TileGroup 游标在调用侧推进后传入。